# DACH News Coverage Analysis — Single Parenthood (2015–2025)

**Media framing analysis** using the GDELT Project — a free, open archive of
news articles indexed worldwide since 2015.

Mirrors the Reddit notebook structure: same keywords, same years, same NLP
pipeline (VADER + XLM-RoBERTa + BERTopic) for direct comparison.

**What this captures:** How journalists *frame* single parenthood in headlines
— which is fundamentally different from how people *discuss* it on Reddit.

## Cell 1 — Install Dependencies

In [1]:
!pip install -q \
    requests pandas matplotlib seaborn \
    "vaderSentiment" \
    "transformers>=4.36.0" "torch>=2.0" "safetensors>=0.4" "protobuf>=3.20" \
    "scikit-learn>=1.0" langdetect scipy nltk \
    "bertopic>=0.16" "sentence-transformers>=2.2" \
    "umap-learn>=0.5" "hdbscan>=0.8" "keybert"

print("All packages installed.")

All packages installed.


## Cell 2 — Imports

In [2]:
import os, json, time, re, html, warnings
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import quote_plus

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from langdetect import detect, LangDetectException
import nltk

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100
print("All imports ready.")

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports ready.


## Cell 3 — Configuration

Same keywords and year range as the Reddit notebook.

In [3]:
# Same keywords as Reddit notebook
KEYWORDS_DE = [
    "Alleinerziehende", "Alleinerziehender", "Einelternfamilie",
    "alleinerziehend", "Alleinerziehenden", "Alleinerzieherin",
    "Sozialmutter", "Solovater", "Solomutter",
]
KEYWORDS_EN = [
    "single parent", "single mother", "single father",
    "single mom", "single dad", "lone parent",
]
KEYWORDS = KEYWORDS_DE + KEYWORDS_EN

YEAR_START = 2020
YEAR_END = 2025
YEARS = list(range(YEAR_START, YEAR_END + 1))

# GDELT settings
GDELT_API_URL = "https://api.gdeltproject.org/api/v2/doc/doc"
GDELT_MAX_RECORDS = 250
GDELT_SOURCE_COUNTRIES = ["GM", "AU", "SZ"]  # Germany, Austria, Switzerland
GDELT_LANGUAGES = ["german", "english"]

# Cache files
RAW_CACHE_FILE = "gdelt_raw_cache.json"
SENTIMENT_CACHE_FILE = "gdelt_sentiment_cache.json"
TOPIC_CACHE_FILE = "gdelt_topic_cache.json"

# Analysis
ROBERTA_BATCH_SIZE = 16
N_TOPICS = 8
MIN_TOPIC_SIZE = 10
MIN_TEXT_LEN_FOR_TOPICS = 30
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
REQUEST_DELAY = 1.0

total = len(YEARS) * len(KEYWORDS) * len(GDELT_SOURCE_COUNTRIES) * len(GDELT_LANGUAGES)
print(f"Config loaded. {total} GDELT queries planned.")

Config loaded. 990 GDELT queries planned.


## Cell 4 — GDELT Data Collection

The GDELT DOC 2.0 API is queried year by year, keyword by keyword,
filtered by DACH source countries and language. Free, no API key.

In [4]:
# Cell 4 - Fetch GDELT articles

def fetch_gdelt(keyword, country, language, start_dt, end_dt):
    kw_q = f'"{keyword}"' if " " in keyword else keyword
    query = f'{kw_q} sourcecountry:{country} lang:{language}'
    params = {"query": query, "mode": "artlist", "maxrecords": GDELT_MAX_RECORDS,
              "format": "json", "startdatetime": start_dt, "enddatetime": end_dt}
    try:
        resp = requests.get(GDELT_API_URL, params=params, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            return [{"id": a.get("url",""), "title": a.get("title",""),
                     "url": a.get("url",""), "domain": a.get("domain",""),
                     "source_country": a.get("sourcecountry",""),
                     "seendate": a.get("seendate",""),
                     "language": a.get("language",""),
                     "gdelt_tone": a.get("tone", 0),
                     "_keyword": keyword, "_year": int(start_dt[:4])}
                    for a in data.get("articles", [])]
    except Exception:
        pass
    return []

if os.path.exists(RAW_CACHE_FILE):
    print(f"Cache found ({RAW_CACHE_FILE}). Loading ...")
    with open(RAW_CACHE_FILE, "r", encoding="utf-8") as f:
        raw_articles = json.load(f)
    print(f"  Loaded {len(raw_articles):,} articles.")
else:
    print("Fetching GDELT articles ...")
    raw_articles = []
    seen = set()
    total = len(YEARS) * len(KEYWORDS) * len(GDELT_SOURCE_COUNTRIES) * len(GDELT_LANGUAGES)
    done = 0
    for year in YEARS:
        s = f"{year}0101000000"
        e = f"{year}1231235959"
        for kw in KEYWORDS:
            for co in GDELT_SOURCE_COUNTRIES:
                for la in GDELT_LANGUAGES:
                    done += 1
                    if done % 25 == 0 or done == total:
                        print(f"  [{done}/{total}] {year} | {kw[:20]:20s} | {co} | {la:7s} | {len(raw_articles):,}")
                    arts = fetch_gdelt(kw, co, la, s, e)
                    for a in arts:
                        if a["url"] not in seen:
                            seen.add(a["url"])
                            raw_articles.append(a)
                    time.sleep(REQUEST_DELAY)
    with open(RAW_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(raw_articles, f, ensure_ascii=False)
    print(f"\nCached {len(raw_articles):,} articles.")

RAW_TOTAL = len(raw_articles)
print(f"\nTotal: {RAW_TOTAL:,} articles")

Fetching GDELT articles ...
  [25/990] 2015 | Alleinerziehenden    | GM | german  | 0
  [50/990] 2015 | Solomutter           | GM | english | 0
  [75/990] 2015 | single mom           | AU | german  | 0
  [100/990] 2016 | Alleinerziehender    | AU | english | 0
  [125/990] 2016 | Alleinerzieherin     | SZ | german  | 0
  [150/990] 2016 | single parent        | SZ | english | 0
  [175/990] 2016 | lone parent          | GM | german  | 0
  [200/990] 2017 | alleinerziehend      | GM | english | 0
  [225/990] 2017 | Solovater            | AU | german  | 0
  [250/990] 2017 | single father        | AU | english | 0
  [275/990] 2018 | Alleinerziehende     | SZ | german  | 0
  [300/990] 2018 | Alleinerziehenden    | SZ | english | 0
  [325/990] 2018 | single parent        | GM | german  | 0
  [350/990] 2018 | single dad           | GM | english | 0
  [375/990] 2019 | Einelternfamilie     | AU | german  | 0
  [400/990] 2019 | Sozialmutter         | AU | english | 0
  [425/990] 2019 | single mothe

## Cell 5 — Preprocessing

In [5]:
# Cell 5 - Preprocessing
def clean_text(t):
    t = html.unescape(t)
    t = re.sub(r"https?://\S+", "", t)
    t = re.sub(r"&\w+;", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def detect_lang(t):
    try: return detect(t)
    except: return "unknown"

df = pd.DataFrame(raw_articles)
if len(df) > 0:
    df["full_text"] = df["title"].fillna("").apply(clean_text)
    df = df[df["full_text"].str.len() >= 15].copy()
    BEFORE_DEDUP = len(df)
    df = df.drop_duplicates(subset="url", keep="first")
    AFTER_DEDUP = len(df)
    df["date"] = pd.to_datetime(df["seendate"], format="%Y%m%dT%H%M%SZ", errors="coerce", utc=True)
    df["year"] = df["_year"]
    print("Detecting languages ...")
    df["detected_language"] = df["full_text"].apply(detect_lang)
    BEFORE_LANG = len(df)
    df = df[df["detected_language"].isin(["de", "en"])].copy()
    AFTER_LANG = len(df)
    df["gdelt_tone"] = pd.to_numeric(df["gdelt_tone"], errors="coerce").fillna(0)
    PREPROCESS_STATS = {"raw": RAW_TOTAL, "dedup": AFTER_DEDUP, "lang": AFTER_LANG, "final": len(df)}
    print(f"\nRaw: {RAW_TOTAL:,} | Dedup: {AFTER_DEDUP:,} | Lang filter: {AFTER_LANG:,} | Final: {len(df):,}")
    print(f"  DE: {(df['detected_language']=='de').sum():,} | EN: {(df['detected_language']=='en').sum():,}")
else:
    print("No data.")

No data.


## Cell 6 — Dataset Report

In [6]:
# Cell 6 - Dataset report
if len(df) > 0:
    sep = "=" * 72
    lines = [sep, "GDELT DATASET REPORT", f"Generated: {datetime.now()}", sep]
    n = len(df)
    lines.append(f"\n  Final: {n:,} articles")
    lines.append("\n-- BY YEAR --")
    for yr in sorted(df["year"].unique()):
        c = (df["year"]==yr).sum()
        lines.append(f"  {int(yr)}: {c:>5,} ({c/n*100:.1f}%)")
    lines.append("\n-- BY COUNTRY --")
    for sc, c in df["source_country"].value_counts().items():
        lines.append(f"  {sc:<20s}: {c:>5,}")
    lines.append("\n-- TOP 15 DOMAINS --")
    for d, c in df["domain"].value_counts().head(15).items():
        lines.append(f"  {d:<35s}: {c:>4,}")
    lines.append("\n-- BY KEYWORD --")
    for kw, c in df["_keyword"].value_counts().items():
        lines.append(f"  '{kw}': {c:>5,}")
    lines.append(f"\n-- GDELT TONE --")
    lines.append(f"  Mean={df['gdelt_tone'].mean():+.2f} Median={df['gdelt_tone'].median():+.2f}")
    lines.append("\n" + sep)
    report = "\n".join(lines)
    print(report)
    with open("gdelt_dataset_report.txt", "w") as f: f.write(report)
    print("\nSaved gdelt_dataset_report.txt")

## Cell 7 — Sentiment Analysis

In [7]:
# Cell 7 - Sentiment
if os.path.exists(SENTIMENT_CACHE_FILE):
    print(f"Cache found. Loading ...")
    with open(SENTIMENT_CACHE_FILE) as f: sc = json.load(f)
    if len(sc.get("vader",[])) != len(df):
        print("  Mismatch. Recomputing."); os.remove(SENTIMENT_CACHE_FILE)
    else:
        df["vader_sentiment"] = sc["vader"]
        df["roberta_sentiment"] = sc["roberta"]
        df["roberta_confidence"] = sc["confidence"]
        print("  Loaded.")

if not os.path.exists(SENTIMENT_CACHE_FILE) and len(df) > 0:
    print("VADER ...")
    vader = SentimentIntensityAnalyzer()
    def vl(t):
        s = vader.polarity_scores(t)["compound"]
        if s >= 0.05: return "positive"
        elif s <= -0.05: return "negative"
        return "neutral"
    df["vader_sentiment"] = df["full_text"].apply(vl)

    print("RoBERTa ...")
    mn = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
    try:
        tok = AutoTokenizer.from_pretrained(mn, local_files_only=True)
        mod = AutoModelForSequenceClassification.from_pretrained(mn, use_safetensors=True, local_files_only=True)
    except:
        try:
            tok = AutoTokenizer.from_pretrained(mn, local_files_only=True)
            mod = AutoModelForSequenceClassification.from_pretrained(mn, local_files_only=True)
        except:
            os.environ.pop("HF_HUB_OFFLINE", None); os.environ.pop("TRANSFORMERS_OFFLINE", None)
            tok = AutoTokenizer.from_pretrained(mn)
            mod = AutoModelForSequenceClassification.from_pretrained(mn, use_safetensors=True)
            os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"

    rob = pipeline("sentiment-analysis", model=mod, tokenizer=tok, truncation=True, max_length=512, device=-1)
    LM = {"positive":"positive","Positive":"positive","negative":"negative","Negative":"negative",
          "neutral":"neutral","Neutral":"neutral","LABEL_0":"negative","LABEL_1":"neutral","LABEL_2":"positive"}
    texts = df["full_text"].tolist()
    rl, rc = [], []
    for i in range(0, len(texts), ROBERTA_BATCH_SIZE):
        batch = texts[i:i+ROBERTA_BATCH_SIZE]
        for r in rob(batch):
            rl.append(LM.get(r["label"], r["label"]))
            rc.append(round(r["score"], 4))
        d = min(i+ROBERTA_BATCH_SIZE, len(texts))
        if (i//ROBERTA_BATCH_SIZE) % 10 == 0 or d == len(texts): print(f"  {d:,}/{len(texts):,}")
    df["roberta_sentiment"] = rl
    df["roberta_confidence"] = rc
    with open(SENTIMENT_CACHE_FILE, "w") as f:
        json.dump({"vader": df["vader_sentiment"].tolist(), "roberta": rl, "confidence": rc}, f)

if "roberta_sentiment" in df.columns:
    df["primary_sentiment"] = df["roberta_sentiment"]
    df["sentiment_score"] = df["primary_sentiment"].map({"positive":1,"neutral":0,"negative":-1})
    print(f"\nDone. {len(df):,} articles.")
    print(df["primary_sentiment"].value_counts().to_string())

## Cell 8 — News Sentiment Report

In [8]:
# Cell 8 - Sentiment report
if len(df) > 0 and "primary_sentiment" in df.columns:
    sep = "=" * 72
    lines = [sep, "NEWS SENTIMENT REPORT", f"Generated: {datetime.now()}", sep]
    n = len(df)
    lines.append("\n-- OVERALL --")
    for l in ["positive","neutral","negative"]:
        c = int((df["primary_sentiment"]==l).sum())
        lines.append(f"  {l.capitalize()}: {c:>5,} ({c/n*100:.1f}%)")
    lines.append(f"  Mean: {df['sentiment_score'].mean():+.3f}")

    lines.append("\n-- GDELT TONE vs ROBERTA --")
    lines.append(f"  GDELT mean: {df['gdelt_tone'].mean():+.3f}")
    lines.append(f"  RoBERTa mean: {df['sentiment_score'].mean():+.3f}")
    r, p = stats.pearsonr(df["gdelt_tone"], df["sentiment_score"])
    lines.append(f"  Pearson r={r:.3f}, p={p:.4f}")

    lines.append("\n-- BY YEAR --")
    for yr in sorted(df["year"].unique()):
        s = df[df["year"]==yr]
        ns = len(s)
        if ns == 0: continue
        pos = (s["primary_sentiment"]=="positive").sum()
        neg = (s["primary_sentiment"]=="negative").sum()
        ms = s["sentiment_score"].mean()
        gt = s["gdelt_tone"].mean()
        lines.append(f"  {int(yr)}: {ns:>5,} articles | {pos/ns*100:.1f}% pos | {neg/ns*100:.1f}% neg | rob={ms:+.3f} gdelt={gt:+.2f}")

    lines.append("\n-- BY COUNTRY --")
    for sc in df["source_country"].value_counts().index:
        s = df[df["source_country"]==sc]
        lines.append(f"  {sc}: {len(s):,} articles, rob={s['sentiment_score'].mean():+.3f}, gdelt={s['gdelt_tone'].mean():+.2f}")

    lines.append("\n-- TOP 10 DOMAINS --")
    for d in df["domain"].value_counts().head(10).index:
        s = df[df["domain"]==d]
        lines.append(f"  {d:<35s}: {len(s):>4,} articles, rob={s['sentiment_score'].mean():+.3f}")

    lines.append("\n" + sep)
    report = "\n".join(lines)
    print(report)
    with open("gdelt_sentiment_report.txt", "w") as f: f.write(report)
    print("\nSaved gdelt_sentiment_report.txt")

## Cell 9 — Statistical Tests

In [9]:
# Cell 9 - Stats
stat_results = {}
if len(df) > 0 and "sentiment_score" in df.columns:
    ym = df.groupby("year")["sentiment_score"].mean()
    ys = sorted(ym.index)
    ms = [ym[y] for y in ys]
    print("=" * 72)
    print("STATISTICAL ANALYSIS")
    print("=" * 72)

    tau, p = stats.kendalltau(range(len(ms)), ms)
    stat_results["kendall"] = {"tau": round(tau,4), "p": round(p,4)}
    print(f"\n1. Kendall: tau={tau:.4f}, p={p:.4f} ({'Sig' if p<0.05 else 'NS'})")

    yg = [g["sentiment_score"].values for _,g in df.groupby("year") if len(g)>0]
    if len(yg)>=2:
        h, p = stats.kruskal(*yg)
    else: h, p = 0, 1
    stat_results["kruskal"] = {"H": round(h,4), "p": round(p,4)}
    print(f"2. Kruskal-Wallis: H={h:.4f}, p={p:.4f} ({'Sig' if p<0.05 else 'NS'})")

    ea = df[df["year"].between(2015,2019)]["sentiment_score"]
    la = df[df["year"].between(2020,2025)]["sentiment_score"]
    if len(ea)>0 and len(la)>0:
        u, p = stats.mannwhitneyu(ea, la, alternative="two-sided")
    else: u, p = 0, 1
    stat_results["mann_whitney"] = {"U": round(u,4), "p": round(p,4)}
    d = "more positive" if la.mean()>ea.mean() else "more negative"
    print(f"3. Mann-Whitney: U={u:.0f}, p={p:.4f} ({d})")

    r, p = stats.pearsonr(df["gdelt_tone"], df["sentiment_score"])
    stat_results["gdelt_roberta"] = {"r": round(r,4), "p": round(p,4)}
    print(f"4. GDELT vs RoBERTa: r={r:.4f}, p={p:.4f}")
    print("=" * 72)

## Cell 10 — Topic Modeling

In [10]:
# Cell 10 - Topics
if len(df) > 0 and "primary_sentiment" in df.columns:
    COK = False
    if os.path.exists(TOPIC_CACHE_FILE):
        with open(TOPIC_CACHE_FILE) as f: tc = json.load(f)
        if len(tc.get("topics",[])) == len(df):
            df["topic_id"] = tc["topics"]; df["topic_label"] = tc["labels"]
            COK = True; print(f"Loaded {df['topic_id'].nunique()} topics.")
    if not COK:
        de_s = set(stopwords.words("german")); en_s = set(stopwords.words("english"))
        ds = {"alleinerziehende","alleinerziehend","single","parent","mother","father",
              "kinder","kind","family","familie","mutter","vater","eltern","mehr","neue","seit"}
        stops = list(de_s | en_s | ds)
        mask = df["full_text"].str.len() >= MIN_TEXT_LEN_FOR_TOPICS
        docs = df.loc[mask, "full_text"].tolist(); idx = df.loc[mask].index.tolist()
        print(f"Topics on {len(docs):,} articles ...")
        try: emb = SentenceTransformer(EMBEDDING_MODEL_NAME, local_files_only=True)
        except:
            os.environ.pop("HF_HUB_OFFLINE",None); os.environ.pop("TRANSFORMERS_OFFLINE",None)
            emb = SentenceTransformer(EMBEDDING_MODEL_NAME)
            os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
        embeddings = emb.encode(docs, show_progress_bar=True, batch_size=32)
        tm = BERTopic(embedding_model=emb,
            umap_model=UMAP(n_components=5, n_neighbors=10, min_dist=0.0, metric="cosine", random_state=42),
            hdbscan_model=HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, min_samples=3, metric="euclidean", prediction_data=True),
            vectorizer_model=CountVectorizer(stop_words=stops, ngram_range=(1,2), min_df=2, max_df=0.85),
            representation_model=[KeyBERTInspired(top_n_words=10), MaximalMarginalRelevance(diversity=0.3)],
            min_topic_size=MIN_TOPIC_SIZE, nr_topics=N_TOPICS, verbose=True)
        topics, _ = tm.fit_transform(docs, embeddings)
        ti = tm.get_topic_info()
        labels = {}
        for _, row in ti.iterrows():
            tid = row["Topic"]
            if tid == -1: labels[tid] = "Unclassified"
            else:
                tw = tm.get_topic(tid)
                labels[tid] = f"T{tid}: {', '.join(w for w,_ in tw[:4])}" if tw else f"T{tid}"
        df["topic_id"] = -1; df.loc[idx, "topic_id"] = topics
        df["topic_label"] = df["topic_id"].map(labels)
        with open(TOPIC_CACHE_FILE, "w") as f:
            json.dump({"topics": df["topic_id"].tolist(), "labels": df["topic_label"].tolist()}, f)
        for _, row in ti.iterrows():
            tw = tm.get_topic(row["Topic"])
            ws = ", ".join(w for w,_ in tw[:8]) if tw else ""
            print(f"  {'UNCLASS' if row['Topic']==-1 else f'T{row[chr(84)+chr(111)+chr(112)+chr(105)+chr(99)]}'} ({row['Count']:,}): {ws}")
    print(f"\n{df['topic_id'].nunique()} topics assigned.")

## Cell 11 — Figure 1: Volume and Sentiment

In [11]:
# Cell 11 - Figure 1
if len(df) > 0 and "sentiment_score" in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0,0]
    pv = df.pivot_table(index="year", columns="source_country", values="url", aggfunc="count", fill_value=0)
    pv.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", edgecolor="white")
    ax.set_title("A) Articles per Year by Country", weight="bold"); ax.set_ylabel("Articles")

    ax = axes[0,1]
    ym = df.groupby("year").agg(rob=("sentiment_score","mean"), gdelt=("gdelt_tone","mean"))
    ax.plot(ym.index, ym["rob"], "o-", color="steelblue", lw=2, label="RoBERTa")
    ax2 = ax.twinx()
    ax2.plot(ym.index, ym["gdelt"], "s--", color="darkorange", lw=2, label="GDELT tone")
    ax.set_title("B) Yearly Sentiment", weight="bold"); ax.set_ylabel("RoBERTa", color="steelblue")
    ax2.set_ylabel("GDELT tone", color="darkorange"); ax.axhline(0, color="grey", ls=":", lw=0.8)
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, fontsize=8)

    ax = axes[1,0]
    ys = df.groupby(["year","primary_sentiment"]).size().unstack(fill_value=0)
    yp = ys.div(ys.sum(axis=1), axis=0)*100
    co = {"positive":"#4CAF50","neutral":"#FFC107","negative":"#F44336"}
    bot = np.zeros(len(yp))
    for l in ["positive","neutral","negative"]:
        if l in yp.columns:
            ax.bar(yp.index, yp[l].values, bottom=bot, label=l.capitalize(), color=co[l], edgecolor="white")
            bot += yp[l].values
    ax.set_title("C) Sentiment Distribution (%)", weight="bold"); ax.legend(fontsize=8)

    ax = axes[1,1]
    td = df["domain"].value_counts().head(10).index
    ds = df[df["domain"].isin(td)].groupby("domain")["sentiment_score"].mean().sort_values()
    bc = ["#4CAF50" if v>0.05 else "#F44336" if v<-0.05 else "#FFC107" for v in ds.values]
    ax.barh(range(len(ds)), ds.values, color=bc)
    ax.set_yticks(range(len(ds))); ax.set_yticklabels(ds.index, fontsize=8)
    ax.set_title("D) Top Domains by Sentiment", weight="bold"); ax.axvline(0, color="grey", ls=":", lw=0.8)

    plt.suptitle("Figure 1 -- News Coverage Overview", fontsize=15, weight="bold", y=1.01)
    plt.tight_layout(); fig.savefig("gdelt_figure1.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Saved gdelt_figure1.png")

## Cell 12 — Figure 2: Topics

In [12]:
# Cell 12 - Figure 2
if len(df) > 0 and "topic_id" in df.columns:
    real = df[df["topic_id"] != -1]
    tids = real["topic_id"].value_counts().head(min(N_TOPICS,8)).index.tolist()
    sl = {}
    for t in tids:
        l = real[real["topic_id"]==t]["topic_label"].iloc[0]
        sl[t] = l[:35]+"..." if len(l)>37 else l
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    ax = axes[0,0]
    yrs = sorted(real["year"].dropna().unique())
    prev = pd.DataFrame(index=yrs, columns=tids, dtype=float)
    for y in yrs:
        yd = real[real["year"]==y]; yn = max(len(yd),1)
        for t in tids: prev.loc[y,t] = (yd["topic_id"]==t).sum()/yn*100
    ax.stackplot(yrs, [prev[t].values for t in tids], labels=[sl[t] for t in tids],
                 colors=plt.cm.Set2(np.linspace(0,1,len(tids))), alpha=0.85)
    ax.set_title("A) Topic Prevalence", weight="bold"); ax.set_ylabel("%")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02,1), fontsize=7)

    ax = axes[0,1]
    ts = real.groupby("topic_id")["sentiment_score"].mean().reindex(tids)
    tc = real["topic_id"].value_counts().reindex(tids)
    bd = pd.DataFrame({"l":[sl[t] for t in tids],"m":ts.values,"c":tc.values}).sort_values("m")
    bc = ["#4CAF50" if v>0.05 else "#F44336" if v<-0.05 else "#FFC107" for v in bd["m"]]
    ax.barh(range(len(bd)), bd["m"], color=bc)
    ax.set_yticks(range(len(bd))); ax.set_yticklabels(bd["l"], fontsize=8)
    ax.set_title("B) Sentiment per Topic", weight="bold"); ax.axvline(0, color="grey", ls=":", lw=0.8)

    ax = axes[1,0]
    ht = real.pivot_table(index="topic_id",columns="year",values="sentiment_score",aggfunc="mean").reindex(tids)
    ht.index = [sl[t] for t in tids]
    sns.heatmap(ht, annot=True, fmt=".2f", cmap="RdYlGn", center=0, ax=ax, linewidths=0.5, annot_kws={"fontsize":7})
    ax.set_title("C) Topic x Year", weight="bold"); ax.set_ylabel("")

    ax = axes[1,1]
    td = []
    for t in tids:
        s = real[real["topic_id"]==t]; ns = max(len(s),1)
        td.append({"p":(s["primary_sentiment"]=="positive").sum()/ns*100,
                   "u":(s["primary_sentiment"]=="neutral").sum()/ns*100,
                   "n":(s["primary_sentiment"]=="negative").sum()/ns*100})
    td = pd.DataFrame(td); x = np.arange(len(tids))
    ax.bar(x, td["p"], label="Positive", color="#4CAF50", edgecolor="white")
    ax.bar(x, td["u"], bottom=td["p"], label="Neutral", color="#FFC107", edgecolor="white")
    ax.bar(x, td["n"], bottom=td["p"]+td["u"], label="Negative", color="#F44336", edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels([f"T{t}" for t in tids], fontsize=8)
    ax.set_title("D) Sentiment Composition", weight="bold"); ax.legend(fontsize=8)

    plt.suptitle("Figure 2 -- Topic Analysis", fontsize=15, weight="bold", y=1.01)
    plt.tight_layout(); fig.savefig("gdelt_figure2.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Saved gdelt_figure2.png")

## Cell 13 — Export

In [13]:
# Cell 13 - Export
if len(df) > 0:
    cols = [c for c in ["url","title","full_text","domain","source_country","seendate",
            "date","year","detected_language","gdelt_tone","_keyword",
            "vader_sentiment","roberta_sentiment","roberta_confidence",
            "primary_sentiment","sentiment_score","topic_id","topic_label"] if c in df.columns]
    df[cols].to_csv("gdelt_full_dataset.csv", index=False)
    print("gdelt_full_dataset.csv")

    ya = df.groupby("year").agg(n=("url","count"), rob=("sentiment_score","mean"),
        gdelt=("gdelt_tone","mean"),
        ppos=("primary_sentiment", lambda x: (x=="positive").mean()*100),
        pneg=("primary_sentiment", lambda x: (x=="negative").mean()*100)).round(3)
    ya.to_csv("gdelt_yearly.csv"); print("gdelt_yearly.csv")

    sm = {"generated": datetime.now().isoformat(), "total": len(df),
          "years": [YEAR_START, YEAR_END], "keywords": KEYWORDS,
          "preprocess": PREPROCESS_STATS,
          "stats": stat_results if "stat_results" in dir() else {},
          "sentiment": {"pos": int((df["primary_sentiment"]=="positive").sum()),
                        "neu": int((df["primary_sentiment"]=="neutral").sum()),
                        "neg": int((df["primary_sentiment"]=="negative").sum()),
                        "rob_mean": round(df["sentiment_score"].mean(),4),
                        "gdelt_mean": round(df["gdelt_tone"].mean(),4)}}
    if "topic_id" in df.columns:
        sm["topics"] = {"n": int(df[df["topic_id"]!=-1]["topic_id"].nunique())}
    with open("gdelt_summary.json","w") as f: json.dump(sm, f, indent=2, default=str)
    print("gdelt_summary.json")

    for fn in ["gdelt_dataset_report.txt","gdelt_sentiment_report.txt","gdelt_figure1.png","gdelt_figure2.png"]:
        print(f"  [{'OK' if os.path.exists(fn) else 'MISS'}] {fn}")
    print("\nAll exported.")